In [1]:
# 路径引导：无论从项目根还是 notebooks/ 目录启动内核，都能找到 src 包
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('项目根：', ROOT)

项目根： D:\agent项目集\端到端电商智能决策平台


# 02 业务分析

核心经营问题的可复现分析：增长、品类结构、复购、配送体验。
交互式版本见 Streamlit 看板（`streamlit run dashboard/app.py`），
文字结论见 `reports/business_insights.md`。

In [2]:
import pandas as pd
import plotly.express as px
from src import config, sql_engine

START, END = config.ANALYSIS_START_DATE, config.ANALYSIS_END_DATE + ' 23:59:59'

## 1. 月度订单与收入趋势

In [3]:
monthly = sql_engine.read_sql(f'''
SELECT strftime('%Y-%m', purchase_ts) AS month,
       COUNT(*) AS orders, ROUND(SUM(payment_total), 2) AS revenue
FROM fct_order
WHERE is_valid_order = 1 AND purchase_ts BETWEEN '{START}' AND '{END}'
GROUP BY month ORDER BY month
''')
fig = px.line(monthly, x='month', y=['orders', 'revenue'],
              title='月度订单与收入（2017-01 ~ 2018-08）')
fig.show()

## 2. 品类 GMV Top 15

In [4]:
cats = sql_engine.read_sql(f'''
SELECT category_name_en AS category, ROUND(SUM(price), 2) AS gmv, COUNT(*) AS lines
FROM fct_order_item
WHERE is_valid_order = 1 AND purchase_ts BETWEEN '{START}' AND '{END}'
GROUP BY category ORDER BY gmv DESC LIMIT 15
''')
fig = px.bar(cats, x='gmv', y='category', orientation='h', title='品类 GMV Top 15')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. 复购率：Olist 的真实短板

In [5]:
repeat = sql_engine.read_sql('''
SELECT CASE WHEN n_orders > 1 THEN '复购客户' ELSE '单次客户' END AS seg,
       COUNT(*) AS customers
FROM (SELECT customer_unique_id, COUNT(*) AS n_orders FROM fct_order
      WHERE is_valid_order = 1 GROUP BY customer_unique_id) t
GROUP BY seg
''')
repeat['share'] = (repeat['customers'] / repeat['customers'].sum()).round(4)
repeat

,seg,customers,share
0,单次客户,92102,0.9696
1,复购客户,2888,0.0304


> 约 97% 的客户只买过一次 —— 这直接决定了后续建模方向：
> 流失预测实际预测的是 30 天复购概率，推荐系统也只有购买行为可用。

## 4. 配送时长与评分

In [6]:
delivery = sql_engine.read_sql('''
SELECT CASE WHEN is_late_delivery = 1 THEN '延迟' ELSE '准时' END AS status,
       ROUND(AVG(review_score), 3) AS avg_review, COUNT(*) AS orders
FROM fct_order
WHERE is_valid_order = 1 AND is_delivered = 1 AND review_score IS NOT NULL
GROUP BY status
''')
delivery

,status,avg_review,orders
0,准时,4.296,87706
1,延迟,2.567,7623
